# AIKO — IndexTTS-2.5 Colab test

Tests `IndexTeam/IndexTTS-2.5` on a Colab NVIDIA GPU through **vLLM-Omni**.

Included:
- Japanese zero-shot voice cloning
- speed control
- emotion-control example using the official vLLM-Omni client
- latency, audio duration, RTF and GPU-memory measurements
- small concurrency benchmark
- interactive Gradio UI

**Important:** IndexTTS-2.5 needs a reference voice. Upload only audio you own or have permission to clone.

Unlike Qwen3-TTS, IndexTTS-2.5's current serving path is not true chunk-by-chunk streaming. The UI is interactive, but playback begins after synthesis finishes.


In [ ]:
!nvidia-smi
import sys
print(sys.version)


## 1. Install


In [ ]:
%pip install -q -U "vllm-omni[indextts2]" huggingface_hub gradio httpx soundfile numpy
!git clone -q --depth 1 https://github.com/vllm-project/vllm-omni.git /content/vllm-omni-src || true


## 2. Optional Google Drive output folder


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
OUT_DIR = Path("/content/drive/MyDrive/AIKO/tts-tests/indextts25")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs:", OUT_DIR)


## 3. Download IndexTTS-2.5 weights


In [ ]:
from huggingface_hub import snapshot_download

MODEL_DIR = "/content/IndexTTS-2.5"
snapshot_download(
    repo_id="IndexTeam/IndexTTS-2.5",
    local_dir=MODEL_DIR,
)
print("Model:", MODEL_DIR)


## 4. Upload a reference voice
Use a clean 5–15 second recording with one speaker and little/no background noise.


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Upload a reference WAV/MP3 file.")

REF_AUDIO = str(Path("/content") / next(iter(uploaded)))
Path(REF_AUDIO).write_bytes(next(iter(uploaded.values())))
print("Reference:", REF_AUDIO)


## 5. Start IndexTTS-2.5 server


In [ ]:
import os, subprocess, time, httpx, vllm_omni
from pathlib import Path

PORT = 8092
BASE = f"http://127.0.0.1:{PORT}"
DEPLOY = Path(vllm_omni.__file__).resolve().parent / "deploy" / "indextts2_5.yaml"
print("Deploy config:", DEPLOY)

log_path = "/content/indextts25_vllm.log"
log = open(log_path, "w")

server = subprocess.Popen(
    [
        "vllm", "serve", MODEL_DIR,
        "--omni",
        "--trust-remote-code",
        "--port", str(PORT),
        "--deploy-config", str(DEPLOY),
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
    env={**os.environ, "PYTHONUNBUFFERED":"1"},
)

deadline = time.time() + 1200
while time.time() < deadline:
    if server.poll() is not None:
        raise RuntimeError(f"Server exited. Check {log_path}")
    try:
        r = httpx.get(f"{BASE}/v1/audio/voices", timeout=5)
        if r.status_code < 500:
            print("Index server ready:", r.status_code)
            break
    except Exception:
        pass
    print("Loading IndexTTS-2.5...", end="\r")
    time.sleep(5)
else:
    raise TimeoutError(f"Server did not become ready. Check {log_path}")


## 6. Generate one Japanese line


In [ ]:
import base64, mimetypes, io, json, time, soundfile as sf
from IPython.display import Audio, display

def audio_data_url(path):
    mime = mimetypes.guess_type(path)[0] or "audio/wav"
    raw = Path(path).read_bytes()
    return f"data:{mime};base64," + base64.b64encode(raw).decode("ascii")

REF_DATA = audio_data_url(REF_AUDIO)

def gpu_used_mib():
    out = subprocess.check_output([
        "nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"
    ], text=True).strip().splitlines()[0]
    return int(out)

def index_tts(text, speed=1.0, output_name="index.wav"):
    payload = {
        "model": MODEL_DIR,
        "input": text,
        "response_format": "wav",
        "speed": float(speed),
        "ref_audio": REF_DATA,
        "extra_params": {
            "lang": "ja",
            "text_normalization": True,
        },
    }

    t0 = time.perf_counter()
    r = httpx.post(f"{BASE}/v1/audio/speech", json=payload, timeout=600)
    r.raise_for_status()
    elapsed = time.perf_counter() - t0

    path = OUT_DIR / output_name
    path.write_bytes(r.content)
    data, sr = sf.read(io.BytesIO(r.content))
    duration = len(data) / sr
    metrics = {
        "latency_s": round(elapsed, 3),
        "audio_s": round(duration, 3),
        "rtf": round(elapsed / duration, 3),
        "gpu_used_mib": gpu_used_mib(),
    }
    return path, metrics

TEXT = "今日はいい天気ですね。いっしょに公園へ行きませんか？"
path, metrics = index_tts(TEXT, output_name="index_japanese_test.wav")
print(metrics)
display(Audio(filename=str(path)))


## 7. AIKO-style benchmark


In [ ]:
PROMPTS = {
    "short_word": "学校",
    "question": "今日は学校で何を勉強しましたか？",
    "natural": "えっ、本当に？それはちょっとびっくりした。でも、きっと大丈夫だよ。",
    "teacher": "この漢字は「閉める」と読みます。窓を閉めてください。",
}

results = []
for name, text in PROMPTS.items():
    path, m = index_tts(text, output_name=f"index_{name}.wav")
    row = {"case": name, "chars": len(text), **m}
    results.append(row)
    print(row)

print(json.dumps(results, ensure_ascii=False, indent=2))


## 8. Native speed-control test


In [ ]:
for speed in [0.8, 1.0, 1.2]:
    path, metrics = index_tts(
        "今日は新しい言葉を三つ勉強しましょう。",
        speed=speed,
        output_name=f"index_speed_{speed}.wav",
    )
    print("speed", speed, metrics)
    display(Audio(filename=str(path)))


## 9. Emotion control — official vLLM-Omni client
This keeps the request format aligned with the current IndexTTS-2.5 recipe.


In [ ]:
CLIENT = "/content/vllm-omni-src/examples/online_serving/text_to_speech/indextts2/speech_client.py"
emotion_out = str(OUT_DIR / "index_happy.wav")

cmd = [
    "python", CLIENT,
    "--api-base", BASE,
    "--model-version", "2.5",
    "--model", MODEL_DIR,
    "--ref-audio", REF_AUDIO,
    "--lang", "ja",
    "--text", "やった！今日は全部正解だったよ！",
    "--emo-vector", "1.0", "0.0", "0.0", "0.0", "0.0", "0.0", "0.0", "0.0",
    "--emo-alpha", "0.8",
    "--output", emotion_out,
]
print("Running official client...")
subprocess.run(cmd, check=True)
display(Audio(filename=emotion_out))


## 10. Concurrency benchmark
Start small; IndexTTS has two model stages sharing the GPU.


In [ ]:
import asyncio

async def one_request(client, i):
    payload = {
        "model": MODEL_DIR,
        "input": f"これは同時リクエスト {i+1} のテストです。",
        "response_format": "wav",
        "speed": 1.0,
        "ref_audio": REF_DATA,
        "extra_params": {"lang":"ja", "text_normalization":True},
    }
    t0 = time.perf_counter()
    r = await client.post(f"{BASE}/v1/audio/speech", json=payload, timeout=600)
    r.raise_for_status()
    return time.perf_counter() - t0

async def bench_concurrency(n):
    async with httpx.AsyncClient() as client:
        t0 = time.perf_counter()
        lat = await asyncio.gather(*[one_request(client, i) for i in range(n)])
        wall = time.perf_counter() - t0
    return {
        "concurrency": n,
        "wall_s": round(wall, 3),
        "avg_latency_s": round(sum(lat)/len(lat), 3),
        "requests_per_s": round(n/wall, 3),
        "gpu_used_mib": gpu_used_mib(),
    }

for n in [1, 2, 4]:
    print(await bench_concurrency(n))


## 11. Interactive Gradio UI
This is interactive live testing, but **not streamed playback**: Index returns the WAV after synthesis.


In [ ]:
import gradio as gr

def index_ui(text, speed):
    if not text.strip():
        return None, "Enter Japanese text."
    name = f"index_ui_{int(time.time()*1000)}.wav"
    path, metrics = index_tts(text.strip(), speed=float(speed), output_name=name)
    return str(path), json.dumps(metrics, ensure_ascii=False)

with gr.Blocks() as demo:
    gr.Markdown("# AIKO — IndexTTS-2.5 Japanese")
    text = gr.Textbox(
        value="こんにちは！今日は何を勉強したいですか？",
        label="Japanese text",
        lines=3,
    )
    speed = gr.Slider(0.5, 2.0, value=1.0, step=0.05, label="Speed")
    go = gr.Button("Speak")
    audio = gr.Audio(label="Generated audio", autoplay=True)
    metrics = gr.Textbox(label="Metrics")
    go.click(index_ui, [text, speed], [audio, metrics])

demo.queue().launch(share=True, debug=False)


## 12. Stop server


In [ ]:
server.terminate()
server.wait(timeout=30)
log.close()
print("Stopped.")
